## setup

In [ ]:
import os, re, cv2, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from typing import Tuple
import logging

## preprocess teknofest data

### setup

In [ ]:
original_csv_path = "./data/teknofest/data-original.csv"
original_images_folder = "./data/teknofest/images/original"
original_recreated_annotations_folder = "./data/teknofest/annotations/original-recreated"
padded_resized_grayscaled_images_folder = "./data/teknofest/images/padded-resized-grayscaled"
padded_resized_grayscaled_annotations_folder = "./data/teknofest/annotations/padded-resized-grayscaled"
padded_resized_csv_path = "./data/teknofest/data-padded-resized.csv"

In [ ]:
df = pd.read_csv(original_csv_path)
df.head()

### update csv file to adapt changes (padding & resizing)

In [4]:
def get_image_size(image_path):
    """Get the width and height of an image."""
    with Image.open(image_path) as img:
        return img.size  # Returns (width, height)

def transform_coordinates(original_width, original_height, target_size=512):
    """
    Calculate transformation parameters for center padding to square and resize to target_size.
    
    Args:
        original_width: Original image width
        original_height: Original image height
        target_size: Final square size (default 512)
    
    Returns:
        scale_factor: Factor to scale coordinates and dimensions
        offset_x: X offset after center padding
        offset_y: Y offset after center padding
    """
    # Step 1: Center padding to square
    max_dim = max(original_width, original_height)
    
    # Calculate padding offsets (how much to add on each side)
    pad_x = (max_dim - original_width) // 2
    pad_y = (max_dim - original_height) // 2
    
    # Step 2: Resize square to target_size
    scale_factor = target_size / max_dim
    
    return scale_factor, pad_x, pad_y

def update_annotations(csv_path, images_folder, output_csv_path, target_size=512):
    """
    Update annotations CSV with transformed coordinates.
    
    Args:
        csv_path: Path to the original CSV file
        images_folder: Path to folder containing original images
        output_csv_path: Path to save updated CSV
        target_size: Final image size (default 512)
    """
    # Load the CSV
    df = pd.read_csv(csv_path)
    
    # Create a copy for modifications
    updated_df = df.copy()
    
    print(f"Processing {len(df)} annotations...")
    
    for idx, row in df.iterrows():
        filename = row['filename']
        image_path = os.path.join(images_folder, filename)
        
        if not os.path.exists(image_path):
            print(f"Warning: Image {filename} not found in {images_folder}")
            continue
        
        # Get original image dimensions
        original_width, original_height = get_image_size(image_path)
        
        # Calculate transformation parameters
        scale_factor, pad_x, pad_y = transform_coordinates(
            original_width, original_height, target_size
        )
        
        # Update horizontal lines (a, b, c)
        for line_name in ['a', 'b', 'c']:
            # Update x coordinate (add padding offset, then scale)
            x_col = f'{line_name}_x'
            if pd.notna(row[x_col]):
                new_x = (row[x_col] + pad_x) * scale_factor
                updated_df.at[idx, x_col] = new_x
            
            # Update y coordinate (add padding offset, then scale)
            y_col = f'{line_name}_y'
            if pd.notna(row[y_col]):
                new_y = (row[y_col] + pad_y) * scale_factor
                updated_df.at[idx, y_col] = new_y
            
            # Update width (just scale, no offset needed)
            width_col = f'{line_name}_width'
            if pd.notna(row[width_col]):
                new_width = row[width_col] * scale_factor
                updated_df.at[idx, width_col] = new_width
        
        # Update vertical line
        if pd.notna(row['line_x']):
            new_line_x = (row['line_x'] + pad_x) * scale_factor
            updated_df.at[idx, 'line_x'] = new_line_x
        
        if pd.notna(row['line_y']):
            new_line_y = (row['line_y'] + pad_y) * scale_factor
            updated_df.at[idx, 'line_y'] = new_line_y
        
        if pd.notna(row['line_height']):
            new_line_height = row['line_height'] * scale_factor
            updated_df.at[idx, 'line_height'] = new_line_height
        
        if (idx + 1) % 100 == 0:
            print(f"Processed {idx + 1}/{len(df)} images...")
    
    # Save updated CSV
    updated_df.to_csv(output_csv_path, index=False)
    print(f"Updated annotations saved to: {output_csv_path}")
    
    return updated_df

def verify_transformation(original_csv, updated_csv, sample_indices=[0, 1, 2]):
    """
    Verify the transformation by comparing a few samples.
    """
    orig_df = pd.read_csv(original_csv)
    updated_df = pd.read_csv(updated_csv)
    
    print("\n=== Transformation Verification ===")
    for idx in sample_indices:
        if idx < len(orig_df):
            print(f"\nSample {idx + 1} (filename: {orig_df.iloc[idx]['filename']}):")
            print("Original coordinates:")
            print(f"  a: x={orig_df.iloc[idx]['a_x']:.2f}, y={orig_df.iloc[idx]['a_y']:.2f}, width={orig_df.iloc[idx]['a_width']:.2f}")
            print(f"  line: x={orig_df.iloc[idx]['line_x']:.2f}, y={orig_df.iloc[idx]['line_y']:.2f}, height={orig_df.iloc[idx]['line_height']:.2f}")
            
            print("Updated coordinates:")
            print(f"  a: x={updated_df.iloc[idx]['a_x']:.2f}, y={updated_df.iloc[idx]['a_y']:.2f}, width={updated_df.iloc[idx]['a_width']:.2f}")
            print(f"  line: x={updated_df.iloc[idx]['line_x']:.2f}, y={updated_df.iloc[idx]['line_y']:.2f}, height={updated_df.iloc[idx]['line_height']:.2f}")


In [5]:
updated_df = update_annotations(original_csv_path, original_images_folder, padded_resized_csv_path)
verify_transformation(original_csv_path, padded_resized_csv_path)

Processing 102 annotations...
Processed 100/102 images...
Updated annotations saved to: ./data/teknofest/data-padded-resized.csv

=== Transformation Verification ===

Sample 1 (filename: 1 (1).jpg):
Original coordinates:
  a: x=1209.00, y=1575.00, width=408.00
  line: x=1620.00, y=178.00, height=1568.00
Updated coordinates:
  a: x=209.83, y=262.50, width=68.00
  line: x=278.33, y=29.67, height=261.33

Sample 2 (filename: 1 (2).jpg):
Original coordinates:
  a: x=1739.00, y=1722.00, width=619.00
  line: x=1730.00, y=213.00, height=1575.00
Updated coordinates:
  a: x=294.53, y=291.65, width=104.84
  line: x=293.01, y=36.08, height=266.75

Sample 3 (filename: 1 (3).jpg):
Original coordinates:
  a: x=362.00, y=713.00, width=150.00
  line: x=513.00, y=44.00, height=905.00
Updated coordinates:
  a: x=168.96, y=355.18, width=70.01
  line: x=239.43, y=42.94, height=422.39


/tmp/ipykernel_91501/1288364243.py:72: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '209.83333333333331' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  updated_df.at[idx, x_col] = new_x
/tmp/ipykernel_91501/1288364243.py:78: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '262.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  updated_df.at[idx, y_col] = new_y
/tmp/ipykernel_91501/1288364243.py:78: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '288.66666666666663' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  updated_df.at[idx, y_col] = new_y
/tmp/ipykernel_91501/1288364243.py:84: FutureWarning: Setting an item of incompa

### draw annotations on images

In [9]:
def draw_horizontal_line(image, x, y, width, color, thickness=2, label="", font_scale=0.8, text_thickness=2):
    """
    Draw a horizontal line with centered label and length at the end.
    
    Args:
        image: OpenCV image array
        x, y: Starting coordinates of the line
        width: Width of the line
        color: Color tuple (B, G, R) for OpenCV
        thickness: Line thickness
        label: Label text (e.g., 'a', 'b', 'c')
        font_scale: Adaptive font scale based on image size
        text_thickness: Adaptive text thickness based on image size
    """
    if pd.isna(x) or pd.isna(y) or pd.isna(width):
        return image
    
    # Convert to integers
    x, y, width = int(x), int(y), int(width)
    
    # Draw the horizontal line
    start_point = (x, y)
    end_point = (x + width, y)
    cv2.line(image, start_point, end_point, color, thickness)
    
    font = cv2.FONT_HERSHEY_SIMPLEX
    
    # Add label at the center of the line
    if label:
        center_x = x + width // 2
        center_y = y - int(15 * font_scale)  # Scale the offset based on font size
        
        # Ensure text doesn't go outside image boundaries
        center_y = max(int(25 * font_scale), center_y)
        
        # Get text size for centering
        (text_width, text_height), baseline = cv2.getTextSize(label, font, font_scale, text_thickness)
        text_x = center_x - text_width // 2
        text_x = max(5, min(text_x, image.shape[1] - text_width - 5))
        
        # Draw label
        cv2.putText(image, label, (text_x, center_y), font, font_scale, color, text_thickness)
    
    # Add length measurement at the end of the line
    length_text = f"{int(width)}"
    end_x = x + width + int(10 * font_scale)  # Scale the offset based on font size
    end_y = y + int(5 * font_scale)  # Scale the offset based on font size
    
    # Ensure text doesn't go outside image boundaries
    (length_width, length_height), baseline = cv2.getTextSize(length_text, font, font_scale, text_thickness)
    end_x = min(end_x, image.shape[1] - length_width - 5)
    end_y = max(length_height + 5, min(end_y, image.shape[0] - 5))
    
    # Draw length
    cv2.putText(image, length_text, (end_x, end_y), font, font_scale, color, text_thickness)
    
    return image

def draw_vertical_line(image, x, y, height, color, thickness=2):
    """
    Draw a vertical line without labels or measurements.
    
    Args:
        image: OpenCV image array
        x, y: Starting coordinates of the line
        height: Height of the line
        color: Color tuple (B, G, R) for OpenCV
        thickness: Line thickness
    """
    if pd.isna(x) or pd.isna(y) or pd.isna(height):
        return image
    
    # Convert to integers
    x, y, height = int(x), int(y), int(height)
    
    # Draw the vertical line only
    start_point = (x, y)
    end_point = (x, y + height)
    cv2.line(image, start_point, end_point, color, thickness)
    
    return image

def annotate_single_image(image_path, row, output_path):
    """
    Annotate a single image with all the lines and measurements.
    
    Args:
        image_path: Path to the input image
        row: Pandas row containing annotation data
        output_path: Path to save the annotated image
    """
    # Load image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Could not load image {image_path}")
        return False
    
    # Calculate adaptive scaling based on image size
    height, width = image.shape[:2]
    max_dimension = max(height, width)
    
    # Adaptive line thickness (your existing logic)
    line_thickness = round(max_dimension / 250)
    line_thickness = max(1, line_thickness)  # Ensure minimum thickness of 1
    
    # Adaptive font scaling - scale font size based on image size
    # Base font scale of 0.8 for ~512px images, scale proportionally
    base_font_scale = 0.8
    base_dimension = 512
    font_scale = base_font_scale * (max_dimension / base_dimension)
    font_scale = max(0.4, min(font_scale, 2.0))  # Clamp between 0.4 and 2.0
    
    # Adaptive text thickness - scale text thickness based on image size
    # Base text thickness of 2 for ~512px images, scale proportionally
    base_text_thickness = 2
    text_thickness = round(base_text_thickness * (max_dimension / base_dimension))
    text_thickness = max(1, text_thickness)  # Ensure minimum thickness of 1
    
    # Define colors (BGR format for OpenCV)
    colors = {
        'a': (0, 255, 0),      # Green
        'b': (255, 0, 0),      # Blue  
        'c': (0, 0, 255),      # Red
        'line': (255, 255, 0)  # Cyan
    }
    
    # Draw vertical line FIRST (so it goes to the back)
    x = row['line_x']
    y = row['line_y']
    height = row['line_height']
    
    image = draw_vertical_line(
        image, x, y, height,
        colors['line'],
        thickness=line_thickness
    )
    
    # Draw horizontal lines (a, b, c) ON TOP of vertical line
    for line_name in ['a', 'b', 'c']:
        x = row[f'{line_name}_x']
        y = row[f'{line_name}_y']
        width = row[f'{line_name}_width']
        
        image = draw_horizontal_line(
            image, x, y, width, 
            colors[line_name], 
            thickness=line_thickness,
            label=line_name,
            font_scale=font_scale,
            text_thickness=text_thickness
        )
    
    # Add KTO value if available with adaptive font scaling
    if 'KTO' in row and pd.notna(row['KTO']):
        kto_text = f"KTO: {row['KTO']:.2f}"
        font = cv2.FONT_HERSHEY_SIMPLEX
        # Scale KTO font size (typically larger than line labels)
        kto_font_scale = font_scale * 1.5  # Make KTO text 50% larger than line labels
        kto_font_scale = min(kto_font_scale, 2.5)  # Cap the maximum size
        kto_thickness = max(text_thickness + 1, 3)  # Make KTO text slightly thicker
        color = (255, 255, 255)  # White
        
        # Position at bottom-right corner with adaptive spacing
        (text_width, text_height), baseline = cv2.getTextSize(kto_text, font, kto_font_scale, kto_thickness)
        margin = int(20 * font_scale)  # Scale margin based on font size
        text_x = image.shape[1] - text_width - margin
        text_y = image.shape[0] - margin
        
        # Draw KTO text
        cv2.putText(image, kto_text, (text_x, text_y), font, kto_font_scale, color, kto_thickness)
    
    # Save annotated image
    success = cv2.imwrite(output_path, image)
    if not success:
        print(f"Error: Could not save image to {output_path}")
        return False
    
    return True

def annotate_images_from_csv(csv_path, images_folder, output_folder):
    """
    Main function to annotate all images based on CSV data.
    
    Args:
        csv_path: Path to the updated CSV file with annotations
        images_folder: Path to folder containing processed images
        output_folder: Path to folder where annotated images will be saved
    """
    # Load CSV
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} annotations from CSV")
    
    # Create output folder if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    successful_annotations = 0
    failed_annotations = 0
    
    print(f"Starting annotation process...")
    print(f"Input folder: {images_folder}")
    print(f"Output folder: {output_folder}")
    print("-" * 50)
    
    for idx, row in df.iterrows():
        filename = row['filename']
        
        # Input and output paths
        input_path = os.path.join(images_folder, filename)
        output_path = os.path.join(output_folder, filename)
        
        # Check if input image exists
        if not os.path.exists(input_path):
            print(f"Warning: Image {filename} not found in {images_folder}")
            failed_annotations += 1
            continue
        
        # Annotate the image
        success = annotate_single_image(input_path, row, output_path)
        
        if success:
            successful_annotations += 1
            if (successful_annotations) % 50 == 0:
                print(f"Progress: {successful_annotations}/{len(df)} images annotated")
        else:
            failed_annotations += 1
            print(f"Failed to annotate: {filename}")
    
    print("-" * 50)
    print(f"Annotation complete!")
    print(f"Successfully annotated: {successful_annotations} images")
    print(f"Failed annotations: {failed_annotations} images")
    print(f"Annotated images saved in: {output_folder}")

#### draw on preprocessed images

In [ ]:
annotate_images_from_csv(padded_resized_csv_path, padded_resized_grayscaled_images_folder, padded_resized_grayscaled_annotations_folder)

Loaded 102 annotations from CSV
Starting annotation process...
Input folder: ./data/teknofest/images/padded-resized-grayscaled
Output folder: ./data/teknofest/annotations/padded-resized-grayscaled
--------------------------------------------------
Progress: 50/102 images annotated
Progress: 100/102 images annotated
--------------------------------------------------
Annotation complete!
Successfully annotated: 102 images
Failed annotations: 0 images
Annotated images saved in: ./data/teknofest/annotations/padded-resized-grayscaled


#### draw on original images (recreate them for visual consistency)

In [ ]:
annotate_images_from_csv(original_csv_path, original_images_folder, original_recreated_annotations_folder)

Loaded 102 annotations from CSV
Starting annotation process...
Input folder: ./data/teknofest/images/original
Output folder: ./data/teknofest/annotations/original-recreated
--------------------------------------------------
Progress: 50/102 images annotated
Progress: 100/102 images annotated
--------------------------------------------------
Annotation complete!
Successfully annotated: 102 images
Failed annotations: 0 images
Annotated images saved in: ./data/teknofest/annotations/original-recreated


## other

### check if every image is grayscale

In [1]:
def is_image_grayscale(image_path):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        for pixel in img.getdata():
            r, g, b = pixel
            if r != g or g != b:
                return False
    return True

def check_folder_grayscale(folder_path):
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            image_path = os.path.join(folder_path, filename)
            if not is_image_grayscale(image_path):
                print(f"{filename} is NOT grayscale.")
                return False
    print("All images are grayscale.")
    return True

In [9]:
check_folder_grayscale("./data/trash/sus/images-padded")

All images are grayscale.


True

### resize masks

#### binary

In [ ]:
def resize_binary_masks(input_folder, output_folder, target_size):
    os.makedirs(output_folder, exist_ok=True)
    
    for filename in os.listdir(input_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)

            with Image.open(input_path) as img:
                img = img.convert('1')
                resized_img = img.resize(target_size, resample=Image.NEAREST)
                resized_img.save(output_path)

In [ ]:
resize_binary_masks("./data/trash/sus/annotations-padded", "./data/trash/sus/annotations-padded-512", (512, 512))

#### 4 classes

In [ ]:
COLOR_TO_LABEL = {
    (0, 0, 0): 0,
    (85, 85, 85): 1,
    (170, 170, 170): 2,
    (255, 255, 255): 3
}

LABEL_TO_COLOR = {
    0: (0, 0, 0),
    1: (85, 85, 85),
    2: (170, 170, 170),
    3: (255, 255, 255)
}

def color_to_label_array(img):
    """Convert RGB image to label map."""
    arr = np.array(img)
    label_arr = np.zeros((arr.shape[0], arr.shape[1]), dtype=np.uint8)

    for color, label in COLOR_TO_LABEL.items():
        mask = np.all(arr == color, axis=-1)
        label_arr[mask] = label
    return label_arr

def label_to_color_image(label_arr):
    """Convert label map to RGB image."""
    rgb = np.zeros((label_arr.shape[0], label_arr.shape[1], 3), dtype=np.uint8)
    for label, color in LABEL_TO_COLOR.items():
        rgb[label_arr == label] = color
    return Image.fromarray(rgb)

def resize_segmentation_masks(input_folder, output_folder, target_size=(512, 512)):
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)

            with Image.open(input_path) as img:
                img = img.convert('RGB')
                label_arr = color_to_label_array(img)

                label_img = Image.fromarray(label_arr)
                resized_label = label_img.resize(target_size, resample=Image.NEAREST)
                resized_rgb = label_to_color_image(np.array(resized_label))
                resized_rgb.save(output_path)

In [ ]:
resize_segmentation_masks("./data/trash/sus/images-padded", "./data/trash/sus/images-padded-512")

### recolor binary masks

In [ ]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
def separate_lung_regions(binary_mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Separate left and right lung regions from a binary mask.
    
    Args:
        binary_mask: Binary mask where lungs are 255 and background is 0
        
    Returns:
        Tuple of (left_lung_mask, right_lung_mask) as binary arrays
    """
    # Find connected components
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary_mask, connectivity=8
    )
    
    # Filter out background (label 0) and small components
    min_area = 1000  # Minimum area for a lung region
    lung_components = []
    
    for i in range(1, num_labels):  # Skip background (label 0)
        area = stats[i, cv2.CC_STAT_AREA]
        if area > min_area:
            lung_components.append({
                'label': i,
                'area': area,
                'centroid': centroids[i]
            })
    
    if len(lung_components) < 2:
        raise ValueError(f"Expected 2 lung regions, found {len(lung_components)}")
    
    # Sort by area (largest first) to handle cases with small artifacts
    lung_components.sort(key=lambda x: x['area'], reverse=True)
    
    # Take the two largest components as the lungs
    lung1, lung2 = lung_components[0], lung_components[1]
    
    # Determine left and right based on x-coordinate of centroids
    # In medical imaging, patient's left lung appears on the right side of the image
    if lung1['centroid'][0] < lung2['centroid'][0]:
        # lung1 is on the left side of image (patient's right lung)
        right_lung_label = lung1['label']
        left_lung_label = lung2['label']
    else:
        # lung1 is on the right side of image (patient's left lung)
        left_lung_label = lung1['label']
        right_lung_label = lung2['label']
    
    # Create separate masks
    left_lung_mask = (labels == left_lung_label).astype(np.uint8) * 255
    right_lung_mask = (labels == right_lung_label).astype(np.uint8) * 255
    
    return left_lung_mask, right_lung_mask

def process_single_mask(input_path: str, output_path: str, 
                       left_color: Tuple[int, int, int] = (85, 85, 85),
                       right_color: Tuple[int, int, int] = (170, 170, 170)) -> bool:
    """
    Process a single binary lung mask and save the colored version.
    
    Args:
        input_path: Path to input binary mask
        output_path: Path to save the colored mask
        left_color: RGB color for left lung (default: (85, 85, 85))
        right_color: RGB color for right lung (default: (170, 170, 170))
        
    Returns:
        True if successful, False otherwise
    """
    try:
        # Read the binary mask
        binary_mask = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
        if binary_mask is None:
            logger.error(f"Could not read image: {input_path}")
            return False
        
        # Separate lung regions
        left_lung_mask, right_lung_mask = separate_lung_regions(binary_mask)
        
        # Create colored output image
        height, width = binary_mask.shape
        colored_mask = np.zeros((height, width, 3), dtype=np.uint8)
        
        # Fill left lung with specified color
        colored_mask[left_lung_mask == 255] = left_color
        
        # Fill right lung with specified color
        colored_mask[right_lung_mask == 255] = right_color
        
        # Convert from RGB to BGR for OpenCV
        colored_mask_bgr = cv2.cvtColor(colored_mask, cv2.COLOR_RGB2BGR)
        
        # Save the result
        success = cv2.imwrite(output_path, colored_mask_bgr)
        if not success:
            logger.error(f"Could not save image: {output_path}")
            return False
            
        return True
        
    except Exception as e:
        logger.error(f"Error processing {input_path}: {str(e)}")
        return False

def process_lung_masks_folder(input_folder: str, output_folder: str,
                             left_color: Tuple[int, int, int] = (85, 85, 85),
                             right_color: Tuple[int, int, int] = (170, 170, 170),
                             file_extensions: Tuple[str, ...] = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')) -> None:
    """
    Process all lung masks in a folder and save colored versions.
    
    Args:
        input_folder: Path to folder containing binary lung masks
        output_folder: Path to folder where colored masks will be saved
        left_color: RGB color for left lung (default: (85, 85, 85))
        right_color: RGB color for right lung (default: (170, 170, 170))
        file_extensions: Tuple of valid file extensions to process
    """
    input_path = Path(input_folder)
    output_path = Path(output_folder)
    
    # Create output directory if it doesn't exist
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Find all image files
    image_files = []
    for ext in file_extensions:
        image_files.extend(input_path.glob(f'*{ext}'))
        image_files.extend(input_path.glob(f'*{ext.upper()}'))
    
    if not image_files:
        logger.warning(f"No image files found in {input_folder}")
        return
    
    logger.info(f"Found {len(image_files)} images to process")
    
    successful = 0
    failed = 0
    
    for img_file in image_files:
        output_file = output_path / img_file.name
        
        if process_single_mask(str(img_file), str(output_file), right_color, left_color):
            successful += 1
            if successful % 50 == 0:  # Progress update every 50 images
                logger.info(f"Processed {successful}/{len(image_files)} images")
        else:
            failed += 1
    
    logger.info(f"Processing complete: {successful} successful, {failed} failed")

In [ ]:
process_lung_masks_folder(
    input_folder="./data/trash/sus/annotations-padded-512",
    output_folder="./data/trash/sus/annotations-padded-512-recolored",
    left_color=(85, 85, 85),      # Left lung color
    right_color=(170, 170, 170)   # Right lung color
)

### add data name prefixes to files

In [ ]:
def add_prefix_to_files(folder_path, prefix):
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)

        if not os.path.isfile(file_path):
            continue

        new_filename = prefix + filename
        new_file_path = os.path.join(folder_path, new_filename)

        os.rename(file_path, new_file_path)

In [ ]:
add_prefix_to_files("./data/trash/shenzhen-montgomery/images-split-gender/shenzhen", "shenzhen-")
add_prefix_to_files("./data/trash/shenzhen-montgomery/images-split-gender/montgomery", "montgomery-")

### new